In [0]:
%sql
-- Min/max/avg for key numeric fields in orders
SELECT
  MIN(order_total) AS min_total, MAX(order_total) AS max_total, AVG(order_total) AS avg_total,
  MIN(discount_amount) AS min_discount, MAX(discount_amount) AS max_discount,
  MIN(shipping_cost) AS min_ship, MAX(shipping_cost) AS max_ship
FROM ecommerce.base.orders;

In [0]:
%sql
-- Discount percent should be between 0 and 0.30 per dictionary — flag violations
SELECT order_item_id, discount_percent
FROM ecommerce.base.order_items
WHERE discount_percent < 0 OR discount_percent > 0.30;

In [0]:
%sql
-- Negative or zero quantity/price checks in order_items
SELECT order_item_id, quantity, unit_price
FROM ecommerce.base.order_items
WHERE quantity <= 0 OR unit_price <= 0;

In [0]:
%sql
-- Distribution shape of order_total: percentiles + standard deviation
SELECT
  MIN(order_total) AS min_val,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY order_total) AS p25,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY order_total) AS median,
  AVG(order_total) AS mean_val,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY order_total) AS p75,
  PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY order_total) AS p90,
  MAX(order_total) AS max_val,
  STDDEV(order_total) AS std_dev
FROM ecommerce.base.orders;

In [0]:
%sql
-- Outlier detection using IQR (Interquartile Range) method
WITH quartiles AS (
  SELECT
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY order_total) AS q1,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY order_total) AS q3
  FROM ecommerce.base.orders
)
SELECT o.order_id, o.order_total
FROM ecommerce.base.orders o, quartiles q
WHERE o.order_total < (q.q1 - 1.5 * (q.q3 - q.q1))
   OR o.order_total > (q.q3 + 1.5 * (q.q3 - q.q1));

In [0]:
%sql
-- Correlation between discount_amount and quantity
SELECT CORR(quantity, discount_amount) AS correlation_qty_discount
FROM ecommerce.base.order_items;